<a href="https://colab.research.google.com/github/jaidatta71/ML---Berkeley/blob/main/Reusable_EDA_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reusable EDA Notebook Template for Machine Learning

This notebook is a reusable template for **exploratory data analysis (EDA)** in machine learning projects.

It is designed for:
- classification
- regression
- tabular datasets
- optional time-based analysis

It includes:
- data loading and structure checks
- target analysis
- missing value analysis
- univariate and bivariate analysis
- correlation and outlier checks
- drift and segment analysis
- quick model-based signal check
- final EDA summary

## 1. Imports and setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. User inputs

In [ ]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------
FILE_PATH = "your_data.csv"      # update this
TARGET_COL = "target"            # update this
PROBLEM_TYPE = "classification"  # options: "classification", "regression"
DATE_COL = None                  # example: "order_date"
ID_COLS = []                     # example: ["customer_id", "order_id"]
DROP_COLS = []                   # known columns to drop
TOP_N_CATEGORIES = 15
SAVE_PLOTS = False
PLOTS_DIR = "eda_plots"

if SAVE_PLOTS:
    Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)

print("Configured FILE_PATH:", FILE_PATH)
print("Configured TARGET_COL:", TARGET_COL)
print("Configured PROBLEM_TYPE:", PROBLEM_TYPE)

## 3. Helper functions

In [ ]:
def save_plot(name):
    if SAVE_PLOTS:
        safe_name = name.replace("/", "_").replace(" ", "_").lower()
        plt.savefig(os.path.join(PLOTS_DIR, f"{safe_name}.png"), bbox_inches="tight", dpi=150)

def display_missing_summary(df):
    missing_df = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isnull().sum().values,
        "missing_pct": (df.isnull().mean() * 100).round(2).values,
        "dtype": df.dtypes.astype(str).values,
        "nunique": df.nunique(dropna=False).values
    }).sort_values(["missing_pct", "nunique"], ascending=[False, False])
    return missing_df

def detect_feature_types(df, target_col, id_cols=None):
    id_cols = id_cols or []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()

    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    analysis_numeric_cols = [c for c in numeric_cols if c not in id_cols]
    analysis_categorical_cols = [c for c in categorical_cols if c not in id_cols]

    return numeric_cols, categorical_cols, datetime_cols, analysis_numeric_cols, analysis_categorical_cols

def is_binary_target(s):
    vals = pd.Series(s).dropna().unique()
    return len(vals) == 2

def quick_encode_for_model(df, target_col, datetime_cols=None):
    datetime_cols = datetime_cols or []
    model_df = df.copy()
    model_df = model_df.drop(columns=datetime_cols, errors="ignore")
    model_df = model_df.dropna(subset=[target_col])

    for col in model_df.columns:
        if col == target_col:
            continue
        if model_df[col].dtype == "object" or str(model_df[col].dtype) in ("category", "bool"):
            model_df[col] = model_df[col].astype(str).fillna("MISSING")
        else:
            model_df[col] = model_df[col].fillna(model_df[col].median())

    X = pd.get_dummies(model_df.drop(columns=[target_col]), drop_first=True)
    y = model_df[target_col]
    return X, y, model_df

def print_header(text):
    print("\n" + "=" * 80)
    print(text)
    print("=" * 80)

## 4. Load data

In [ ]:
df = pd.read_csv(FILE_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.sample(min(5, len(df)), random_state=RANDOM_STATE))

## 5. Basic structure review

In [ ]:
print_header("DATAFRAME INFO")
df.info()

print_header("NUMERIC DESCRIBE")
display(df.describe().T)

print_header("ALL COLUMN DESCRIBE")
display(df.describe(include="all").T)

print_header("DUPLICATE ROWS")
print("Duplicate rows:", df.duplicated().sum())

print_header("COLUMN SUMMARY")
dtype_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "null_count": df.isnull().sum().values,
    "null_pct": (df.isnull().mean() * 100).round(2).values,
    "nunique": df.nunique(dropna=False).values
})
display(dtype_summary.sort_values(["dtype", "null_pct"], ascending=[True, False]))

## 6. Basic cleaning and type handling

In [ ]:
df.columns = df.columns.str.strip()

drop_existing = [c for c in DROP_COLS if c in df.columns]
if drop_existing:
    df = df.drop(columns=drop_existing)

if DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

print("Updated shape:", df.shape)
print("Columns:", list(df.columns)[:20], "..." if len(df.columns) > 20 else "")

## 7. Target analysis

In [ ]:
assert TARGET_COL in df.columns, f"{TARGET_COL} not found in dataset"

print_header("TARGET DISTRIBUTION")
display(df[TARGET_COL].value_counts(dropna=False))
display((df[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).round(2))

plt.figure()
if PROBLEM_TYPE == "classification":
    sns.countplot(x=TARGET_COL, data=df)
    plt.title("Target Class Distribution")
else:
    sns.histplot(df[TARGET_COL].dropna(), kde=True)
    plt.title("Target Distribution")
plt.show()
save_plot("target_distribution")

if PROBLEM_TYPE == "regression":
    plt.figure()
    sns.boxplot(x=df[TARGET_COL])
    plt.title("Target Boxplot")
    plt.show()
    save_plot("target_boxplot")

## 8. Missing value analysis

In [ ]:
missing_df = display_missing_summary(df)

print_header("MISSING VALUE SUMMARY")
display(missing_df[missing_df["missing_count"] > 0])

plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()
save_plot("missing_values_heatmap")

missing_plot = missing_df[missing_df["missing_count"] > 0]
if not missing_plot.empty:
    plt.figure(figsize=(10, max(4, len(missing_plot) * 0.3)))
    sns.barplot(data=missing_plot, x="missing_pct", y="column")
    plt.title("Missing Percentage by Column")
    plt.show()
    save_plot("missing_percentage_by_column")

print_header("MISSINGNESS VS TARGET")
for col in df.columns:
    if col != TARGET_COL and df[col].isnull().sum() > 0:
        miss_flag = df[col].isnull().astype(int)
        if PROBLEM_TYPE == "classification":
            print(f"\nColumn: {col}")
            display(df.groupby(miss_flag)[TARGET_COL].mean().rename("target_rate"))
        else:
            print(f"\nColumn: {col}")
            display(df.groupby(miss_flag)[TARGET_COL].agg(["count", "mean", "median"]))

## 9. Feature type detection

In [ ]:
numeric_cols, categorical_cols, datetime_cols, analysis_numeric_cols, analysis_categorical_cols = detect_feature_types(
    df, TARGET_COL, ID_COLS
)

print("Numeric columns:", len(analysis_numeric_cols))
print("Categorical columns:", len(analysis_categorical_cols))
print("Datetime columns:", len(datetime_cols))

display(pd.DataFrame({
    "numeric_cols": pd.Series(analysis_numeric_cols),
    "categorical_cols": pd.Series(analysis_categorical_cols),
    "datetime_cols": pd.Series(datetime_cols)
}))

## 10. Univariate analysis - numeric features

In [ ]:
for col in analysis_numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(df[col].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Histogram: {col}")

    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"Boxplot: {col}")

    plt.tight_layout()
    plt.show()
    save_plot(f"numeric_univariate_{col}")

## 11. Univariate analysis - categorical features

In [ ]:
for col in analysis_categorical_cols:
    vc = df[col].value_counts(dropna=False).head(TOP_N_CATEGORIES)

    plt.figure(figsize=(10, 4))
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Top Categories: {col}")
    plt.ylabel("Count")
    plt.show()
    save_plot(f"categorical_univariate_{col}")

    print(f"\nTop values for {col}")
    display(df[col].value_counts(dropna=False).head(TOP_N_CATEGORIES))

## 12. Bivariate analysis - numeric features vs target

In [ ]:
for col in analysis_numeric_cols:
    if PROBLEM_TYPE == "classification":
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))

        sns.boxplot(x=TARGET_COL, y=col, data=df, ax=axes[0])
        axes[0].set_title(f"{col} vs Target")

        sns.kdeplot(data=df, x=col, hue=TARGET_COL, fill=True, ax=axes[1])
        axes[1].set_title(f"KDE by Target: {col}")

        sns.violinplot(x=TARGET_COL, y=col, data=df, ax=axes[2])
        axes[2].set_title(f"Violin Plot: {col}")

        plt.tight_layout()
        plt.show()
        save_plot(f"numeric_vs_target_{col}")

        summary = df.groupby(TARGET_COL)[col].agg(["count", "mean", "median", "std", "min", "max"])
        print(f"\nSummary stats for {col} by target")
        display(summary)

    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        sns.scatterplot(data=df, x=col, y=TARGET_COL, alpha=0.5, ax=axes[0])
        axes[0].set_title(f"{col} vs {TARGET_COL}")

        sns.regplot(data=df, x=col, y=TARGET_COL, scatter_kws={"alpha": 0.3}, ax=axes[1])
        axes[1].set_title(f"Regression trend: {col} vs {TARGET_COL}")

        plt.tight_layout()
        plt.show()
        save_plot(f"numeric_vs_target_{col}")

## 13. Bivariate analysis - categorical features vs target

In [ ]:
for col in analysis_categorical_cols:
    if PROBLEM_TYPE == "classification":
        ctab = pd.crosstab(df[col], df[TARGET_COL], normalize="index") * 100
        ctab = ctab.sort_values(by=ctab.columns[-1], ascending=False).head(TOP_N_CATEGORIES)

        ctab.plot(kind="bar", stacked=True, figsize=(10, 5))
        plt.title(f"{col} vs Target (%)")
        plt.ylabel("Percentage")
        plt.xticks(rotation=45, ha="right")
        plt.legend(title=TARGET_COL)
        plt.show()
        save_plot(f"categorical_vs_target_{col}")

        target_by_cat = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False).head(TOP_N_CATEGORIES)
        print(f"\nTarget rate by {col}")
        display(target_by_cat)

    else:
        top_cats = df[col].value_counts().index[:TOP_N_CATEGORIES]
        temp = df[df[col].isin(top_cats)]
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=temp, x=col, y=TARGET_COL)
        plt.title(f"{TARGET_COL} by {col}")
        plt.xticks(rotation=45, ha="right")
        plt.show()
        save_plot(f"categorical_vs_target_{col}")

        agg = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False).head(TOP_N_CATEGORIES)
        display(agg)

## 14. Correlation analysis

In [ ]:
if len(analysis_numeric_cols) > 0:
    corr_cols = analysis_numeric_cols + ([TARGET_COL] if TARGET_COL in df.columns else [])
    corr_matrix = df[corr_cols].corr(numeric_only=True)

    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.show()
    save_plot("correlation_heatmap")

    if TARGET_COL in corr_matrix.columns:
        print_header("CORRELATION WITH TARGET")
        display(corr_matrix[TARGET_COL].sort_values(ascending=False))

## 15. Outlier analysis

In [ ]:
outlier_summary = []

for col in analysis_numeric_cols:
    series = df[col].dropna()
    if len(series) == 0:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = ((series < lower) | (series > upper)).sum()
    outlier_pct = round(outlier_count / len(series) * 100, 2)

    outlier_summary.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("outlier_pct", ascending=False)
display(outlier_df)

## 16. Skewness analysis

In [ ]:
if analysis_numeric_cols:
    skew_df = pd.DataFrame({
        "column": analysis_numeric_cols,
        "skewness": [df[c].dropna().skew() for c in analysis_numeric_cols]
    }).sort_values("skewness", ascending=False)

    display(skew_df)

## 17. High-cardinality categorical review

In [ ]:
if analysis_categorical_cols:
    cardinality_df = pd.DataFrame({
        "column": analysis_categorical_cols,
        "nunique": [df[c].nunique(dropna=False) for c in analysis_categorical_cols]
    }).sort_values("nunique", ascending=False)

    display(cardinality_df)

    high_card_cols = cardinality_df[cardinality_df["nunique"] > 50]["column"].tolist()
    print("High-cardinality categorical columns:", high_card_cols)
else:
    high_card_cols = []

## 18. Time-based analysis (optional)

In [ ]:
if DATE_COL and DATE_COL in df.columns:
    temp = df.dropna(subset=[DATE_COL]).copy()
    temp["year_month"] = temp[DATE_COL].dt.to_period("M").astype(str)

    plt.figure(figsize=(12, 5))
    temp.groupby("year_month")[TARGET_COL].mean().plot(marker="o")
    plt.title("Target Trend Over Time")
    plt.xticks(rotation=45)
    plt.show()
    save_plot("target_trend_over_time")

    plt.figure(figsize=(12, 5))
    temp.groupby("year_month").size().plot(marker="o")
    plt.title("Record Count Over Time")
    plt.xticks(rotation=45)
    plt.show()
    save_plot("record_count_over_time")

    for col in analysis_numeric_cols[:5]:
        plt.figure(figsize=(12, 5))
        temp.groupby("year_month")[col].mean().plot(marker="o")
        plt.title(f"Mean of {col} Over Time")
        plt.xticks(rotation=45)
        plt.show()
        save_plot(f"feature_drift_{col}")
else:
    print("DATE_COL not configured or not found. Skipping time-based analysis.")

## 19. Segment analysis

In [ ]:
candidate_segments = [c for c in analysis_categorical_cols if df[c].nunique(dropna=False) <= TOP_N_CATEGORIES]

for seg_col in candidate_segments[:5]:
    plt.figure(figsize=(10, 5))
    if PROBLEM_TYPE == "classification":
        rate = df.groupby(seg_col)[TARGET_COL].mean().sort_values(ascending=False)
        sns.barplot(x=rate.values, y=rate.index)
        plt.title(f"Target Rate by {seg_col}")
    else:
        avg = df.groupby(seg_col)[TARGET_COL].mean().sort_values(ascending=False)
        sns.barplot(x=avg.values, y=avg.index)
        plt.title(f"Mean {TARGET_COL} by {seg_col}")
    plt.show()
    save_plot(f"segment_analysis_{seg_col}")

## 20. Statistical tests

In [ ]:
if PROBLEM_TYPE == "classification" and is_binary_target(df[TARGET_COL]):
    print_header("MANN-WHITNEY U TEST FOR NUMERIC FEATURES")
    results = []
    unique_vals = sorted(df[TARGET_COL].dropna().unique())
    for col in analysis_numeric_cols[:20]:
        grp0 = df[df[TARGET_COL] == unique_vals[0]][col].dropna()
        grp1 = df[df[TARGET_COL] == unique_vals[1]][col].dropna()
        if len(grp0) > 0 and len(grp1) > 0:
            stat, p = stats.mannwhitneyu(grp0, grp1, alternative="two-sided")
            results.append((col, p))
    display(pd.DataFrame(results, columns=["feature", "p_value"]).sort_values("p_value"))

    print_header("CHI-SQUARE TEST FOR CATEGORICAL FEATURES")
    chi_results = []
    from scipy.stats import chi2_contingency

    for col in analysis_categorical_cols[:20]:
        table = pd.crosstab(df[col], df[TARGET_COL])
        if table.shape[0] > 1 and table.shape[1] > 1:
            stat, p, dof, exp = chi2_contingency(table)
            chi_results.append((col, p))
    display(pd.DataFrame(chi_results, columns=["feature", "p_value"]).sort_values("p_value"))
else:
    print("Skipping statistical tests or target is not binary classification.")

## 21. Leakage review checklist

In [ ]:
print("""
Review these manually for each feature:
1. Is the feature available at prediction time?
2. Was the feature created after the outcome happened?
3. Is the feature derived directly from the target?
4. Does the feature use future rolling windows?
5. Is the feature a proxy for the final outcome?
6. Does the feature contain post-event status values?

Recommended manual audit table:
| feature | source | available_at_prediction | leakage_risk | action |
""")

## 22. Quick model-based feature importance proxy

In [ ]:
X_proxy, y_proxy, proxy_df = quick_encode_for_model(df, TARGET_COL, datetime_cols=datetime_cols)

if len(X_proxy.columns) > 0:
    if PROBLEM_TYPE == "classification":
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        )
    else:
        model = RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    model.fit(X_proxy, y_proxy)
    feat_imp = pd.Series(model.feature_importances_, index=X_proxy.columns).sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 6))
    feat_imp.sort_values().plot(kind="barh")
    plt.title("Top 20 Feature Importance (Proxy Model)")
    plt.show()
    save_plot("proxy_feature_importance")

    display(feat_imp.to_frame("importance"))
else:
    print("No features available for proxy model.")

## 23. Optional pairplot for smaller datasets

In [ ]:
sample_cols = analysis_numeric_cols[:4]
if len(sample_cols) >= 2 and df.shape[0] <= 5000:
    if PROBLEM_TYPE == "classification":
        sns.pairplot(df[sample_cols + [TARGET_COL]].dropna(), hue=TARGET_COL)
    else:
        sns.pairplot(df[sample_cols + [TARGET_COL]].dropna())
    plt.show()
    save_plot("pairplot")
else:
    print("Skipping pairplot due to dataset size or insufficient numeric columns.")

## 24. Sanity-check model

In [ ]:
X, y, model_df = quick_encode_for_model(df, TARGET_COL, datetime_cols=datetime_cols)

if len(X.columns) > 0:
    stratify_y = y if (PROBLEM_TYPE == "classification" and is_binary_target(y)) else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify_y
    )

    if PROBLEM_TYPE == "classification":
        clf = RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"
        )
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        print_header("CLASSIFICATION REPORT")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred))

    else:
        reg = RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        reg.fit(X_train, y_train)
        y_pred = reg.predict(X_test)

        print_header("REGRESSION METRICS")
        print("MAE:", mean_absolute_error(y_test, y_pred))
        print("R2 :", r2_score(y_test, y_pred))
else:
    print("No features available for sanity-check model.")

## 25. Final EDA summary

In [ ]:
summary_points = []

summary_points.append(f"Dataset shape: {df.shape}")
summary_points.append(f"Target column: {TARGET_COL}")
summary_points.append(f"Problem type: {PROBLEM_TYPE}")
summary_points.append(f"Numeric columns: {len(analysis_numeric_cols)}")
summary_points.append(f"Categorical columns: {len(analysis_categorical_cols)}")
summary_points.append(f"Datetime columns: {len(datetime_cols)}")
summary_points.append(f"Duplicate rows: {df.duplicated().sum()}")

high_missing = missing_df[missing_df["missing_pct"] > 30]["column"].tolist()
summary_points.append(f"Columns with >30% missing: {high_missing}")

summary_points.append(f"High-cardinality categorical columns: {high_card_cols}")

print_header("EDA SUMMARY")
for point in summary_points:
    print("-", point)

print("\nRecommended next steps:")
print("""
1. Review and remove leakage-prone columns
2. Decide missing value treatment strategy
3. Encode categorical columns appropriately
4. Address class imbalance if needed
5. Transform highly skewed features if needed
6. Review outlier handling approach
7. Build time-aware split if data is temporal
8. Convert stable EDA findings into preprocessing pipeline steps
9. Document assumptions and exclusions
""")

## 26. Notes

Use this notebook for:
- first-pass data understanding
- stakeholder discussions
- feature review
- surfacing data quality issues
- planning preprocessing and feature engineering

After EDA, move stable logic into:
- reusable Python scripts
- SQL transformations
- pipeline jobs
- feature store or curated data layer